# Clock - NHATS (Bone et al. 2025)

**Spec:** PLAN.md section H. **Runs on:** Colab, **no Drive** (NHATS must not go to Kaggle or any AI assistant; do NOT use Colab's Gemini here).

| Part | What | Status |
|---|---|---|
| A | Label table, person-level split, cleaning check (H1-H3) | **ready to run** |
| B | Crop-Net training + multi-step crop + crop QC (H4-H6) | not written yet |
| C | ViT clock score / dementia model (H7) | not written yet |

Colab's disk is wiped on every disconnect, so **each new session** drag and drop these into the *Files* panel (`/content`):

| File (made on your PC in `D:\Alzheimers\colab_upload\`) | Size |
|---|---|
| `clock_code.tar` (`common/`, `clock/`) | tiny |
| `clock_data.tar` (all image zips + `survey/`) | ~1.9 GB |

A GPU is only needed for parts B and C.

# Part A - data prep (H1-H3)

In [ ]:
# Run in EVERY new session: unpack what you uploaded into /content (see the table above)
import os, sys, subprocess
os.makedirs('/content/Alzheimers', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/clock_code.tar', '-C', '/content/Alzheimers'], check=True)
subprocess.run(['tar', '-xf', '/content/clock_data.tar', '-C', '/content/data'], check=True)   # creates /content/data/NHATS
ROOT = '/content/Alzheimers'
NHATS = '/content/data/NHATS'    # image zips here, survey/ inside
OUT = '/content/out'             # Colab temporary disk; wiped on disconnect, so download what you need
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
from clock import labels, split, preprocess
from common.runlog import RunLog
log = RunLog(OUT + '/clock.log')   # appends to a text file on Colab's temporary disk

## 1. Label table (H1)
Expect 54,027 images indexed (Bone et al.).

In [ ]:
with log.tee('H1 label table'):
    tab = labels.build_label_table(NHATS, NHATS + '/survey')
    labels.summarize(tab)


> **SEND BACK TO CLAUDE:**
> the full printed block from step 1 (image counts, exclusion reasons, score counts, dementia %).
> (Expect 54,027 images indexed.)

## 2. Split by person (H2)
Each SPID in exactly one of train / val / test. The last line must print 0.

In [ ]:
with log.tee('H2 split'):
    tab = split.split_by_spid(tab, val_images=1000, test_frac=0.10, seed=0)
    split.report(tab)
    tab.to_csv(OUT + '/split.csv', index=False)   # temporary disk; re-created by re-running step 2


> **SEND BACK TO CLAUDE:**
> the split table from step 2, including the last line `SPIDs in more than one split: 0`.

## 3. Cleaning check (H3)
Shows 8 random clocks: original | after invert + name-box removal + rough crop. Look for (a) name boxes gone, (b) no clock strokes erased, (c) crop contains the whole clock. Tell Claude only what you saw, e.g. '7 of 8 fine, 1 clock lost its outline'.

In [ ]:
import zipfile, matplotlib.pyplot as plt
rows = tab[tab.split != 'excluded'].sample(8, random_state=1)
fig, ax = plt.subplots(8, 3, figsize=(9, 24))
for k, (_, r) in enumerate(rows.iterrows()):
    with zipfile.ZipFile(f'{NHATS}/{r.zip_name}') as z:
        img = preprocess.load_tif(z.read(r.member_path))
    inv = preprocess.invert(img)
    clean = preprocess.remove_name_boxes(inv)
    crop = preprocess.crop_to_mass(clean)
    for a, im, t in zip(ax[k], (img, clean, crop), ('original', 'boxes removed', 'rough crop')):
        a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

> **SEND BACK TO CLAUDE:**
> ONE sentence from you, typed, not the images: e.g. '7 of 8 fine; name boxes gone; 1 clock outline lost'.
> Do not paste or upload the image grid (NHATS data).

## Download the log
Nothing is on Drive. Paste results to Claude; this downloads the text log (aggregate counts only) to your PC. Move it to `D:\Alzheimers\results\`.

In [ ]:
from google.colab import files
files.download(OUT + '/clock.log')

# Part B - Crop-Net (H4-H6)

*Not written yet.* Will train on synthetic pages, write a checkpoint to `/content/out` every epoch, and offer a download cell. After a disconnect: upload the last checkpoint from `D:\Alzheimers\models\` and resume.

# Part C - ViT (H7)

*Not written yet.* Two phases (head, then full body); checkpoint downloaded to `D:\Alzheimers\models\` after each phase, re-uploaded to resume.